In [6]:
# Core
import os, json, math, random, numpy as np, pandas as pd
import sys, importlib
# import text_encoder...

from collections import defaultdict, Counter

# Torch / HF
import torch
from transformers import AutoModel, AutoTokenizer
from tensor2tensor.data_generators import text_encoder
# Clustering & metrics
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score as ARI
from sklearn.metrics import normalized_mutual_info_score as NMI
from sklearn.metrics import homogeneity_completeness_v_measure as HCV
from sklearn.metrics import fowlkes_mallows_score as FMI
from scipy.optimize import linear_sum_assignment

# Viz
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# Repro
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

DEVICE = "cpu"   # keep CPU for evaluation (deterministic & simple)

In [4]:
semeval_holdout = pd.read_parquet("../data/wsd_data/semeval_holdout.parquet")
semeval_holdout.sample(10)

,lemma,sense_id,left_context,target_word,right_context,wsd_source,tokens,target_idx,xmlr_wp_span,labert_wp_span,xmlr_wp_len,labert_wp_len,tokens_lemma,target_idx_lemma,xmlr_wp_span_lemma,labert_wp_span_lemma,xmlr_wp_len_lemma,labert_wp_len_lemma
477,itero,I,quae dicta sunt de oratione dilucida cadunt in...,iterata,ponantur constructioque verborum tum coniuncti...,semeval_wsd,"[quae, dicta, sunt, de, oratione, dilucida, ca...",88,"[149, 150, 151]","[118, 119]",3,2,"[quae, dicta, sunt, de, oratione, dilucida, ca...",88,"[149, 150]","[118, 119]",2,2
157,consilium,I,ad frugem vitae melioris se erigat et optimi m...,consilio,ut dominus patriarcha dominus quoque rainaldus...,semeval_wsd,"[ad, frugem, vitae, melioris, se, erigat, et, ...",84,[139],[104],1,1,"[ad, frugem, vitae, melioris, se, erigat, et, ...",84,[139],[104],1,1
337,hostis,II,adversus nos sacer occupatur mons esquilias vi...,hostes,est quid tandem privatae res vestrae quo statu...,semeval_wsd,"[adversus, nos, sacer, occupatur, mons, esquil...",83,[151],[113],1,1,"[adversus, nos, sacer, occupatur, mons, esquil...",83,"[151, 152]",[113],2,1
273,credo,III,mortalem esse docens animum quoque dicere cred...,crede,animam quoque diffundi multoque perire ocius e...,semeval_wsd,"[mortalem, esse, docens, animum, quoque, dicer...",92,[160],[126],1,1,"[mortalem, esse, docens, animum, quoque, dicer...",92,[160],[126],1,1
30,adsumo,I,tria remedia vomitus alvi ductionis vini per t...,adsumere,eadem in quartana facienda sunt sed cum haec t...,semeval_wsd,"[tria, remedia, vomitus, alvi, ductionis, vini...",82,"[135, 136, 137]","[110, 111]",3,2,"[tria, remedia, vomitus, alvi, ductionis, vini...",82,"[135, 136, 137]","[110, 111]",3,2
370,humanitas,III,brevi potitus est antonius desperatis rebus cu...,humanitatis,multis ignovit a quibus saepe graviter laesus ...,semeval_wsd,"[brevi, potitus, est, antonius, desperatis, re...",80,"[163, 164]",[115],2,1,"[brevi, potitus, est, antonius, desperatis, re...",80,"[163, 164]",[115],2,1
259,credo,III,blanditurque quibus haeret amoenitas illa uos ...,credenti,tempore mortis est dei lex prima fundamentum p...,semeval_wsd,"[blanditurque, quibus, haeret, amoenitas, illa...",79,"[137, 138]","[107, 108]",2,2,"[blanditurque, quibus, haeret, amoenitas, illa...",79,[137],[107],1,1
119,cohors,I,quid deinde helisaeus consectator vitae huius ...,cohors,sancta secreti fluminis ripis velut quibusdam ...,semeval_wsd,"[quid, deinde, helisaeus, consectator, vitae, ...",69,"[135, 136, 137]","[102, 103]",3,2,"[quid, deinde, helisaeus, consectator, vitae, ...",69,"[135, 136, 137]","[102, 103]",3,2
23,adsumo,I,salvius et lupicinus scutarius unus alter e sc...,adsumpta,fiducia restiterunt aequataque parumper proeli...,semeval_wsd,"[salvius, et, lupicinus, scutarius, unus, alte...",86,"[178, 179, 180]","[134, 135]",3,2,"[salvius, et, lupicinus, scutarius, unus, alte...",86,"[178, 179, 180]","[134, 135]",3,2
454,itero,I,laudemus totiens dignum laudibus et dicamus ta...,iterat,sua omnes isti levi temporis impensa invenerun...,semeval_wsd,"[laudemus, totiens, dignum, laudibus, et, dica...",81,"[135, 136]","[104, 105]",2,2,"[laudemus, totiens, dignum, laudibus, et, dica...",81,"[135, 136]","[104, 105]",2,2


In [9]:
print(semeval_holdout.sample(5))

         lemma sense_id                                       left_context  \
73      cohors       II  cum ad nos cotylam mitteret ornamentum atque a...   
414  humanitas       IV  liber iii i marcus sequar igitur ut institui d...   
394  humanitas        I  maluit tamen divinitatem coniungere humanitati...   
277      credo        V  proxima castra die cedente locantur utrimque n...   
399  humanitas      III  tam longe lateque dispersum quo bello omnes ge...   

    target_word                                      right_context  \
73      cohorti  praetoriae praemia agrumque dederitis iis etia...   
414  humanitate  atticus sane gaudeo quod te interpellavi quoni...   
394  humanitate  latere diabolo ut cum praeter ius et fas diabo...   
277      credit  inisse fagam coenantem crapula somno aggravat ...   
399  humanitate  quae breviter qualia sint in cn pompeio consid...   

      wsd_source                                             tokens  \
73   semeval_wsd  [cum, ad, nos, cotyla

In [5]:
# Your processed SemEval holdout with spans & tokens already computed
# Must contain columns:
#   lemma, sense_id, tokens, tokens_lemma,
#   xmlr_wp_span, xmlr_wp_span_lemma,
#   labert_wp_span, labert_wp_span_lemma
semeval_holdout = semeval_holdout.copy().reset_index(drop=True)

# Select the 8 held-out lemmas
heldout_lemmata = sorted(semeval_holdout["lemma"].str.replace(r"\d+$","", regex=True).unique())
print("Held-out lemmas:", heldout_lemmata, " (n=", len(heldout_lemmata), ")")


Held-out lemmas: ['adsumo', 'cohors', 'consilium', 'consul', 'credo', 'hostis', 'humanitas', 'itero']  (n= 8 )


In [7]:
# load xlm-base
MODEL_ID = "xlm-roberta-base"
tokenizer_xlmr = AutoTokenizer.from_pretrained(MODEL_ID)
model_xlmr     = AutoModel.from_pretrained(
                MODEL_ID,
                output_hidden_states=True,
                output_attentions=True
            ).to(DEVICE).eval()

In [8]:
base_path = "/srv/models/latin-bert"
# Initialize the tokenizer with the vocab.txt file and the encoder
vocab_file_path = "/srv/models/latin-bert/vocab.txt" # "/Users/vojtechkase/Projects/latin-bert/models/latin_bert/vocab.txt"  # Update this path
subword_tokenizer_path = "/srv/models/latin-bert/latin.subword.encoder"
# Update this path
encoder = text_encoder.SubwordTextEncoder(subword_tokenizer_path)

spec = importlib.util.spec_from_file_location(
    "latin_tokenizer",
    os.path.join(base_path, "latin_tokenizer.py")
)
latin_tokenizer = importlib.util.module_from_spec(spec)
sys.modules["latin_tokenizer"] = latin_tokenizer   # ensure it's registered
spec.loader.exec_module(latin_tokenizer)

LatinTokenizer = latin_tokenizer.LatinTokenizer
tokenizer_labert = LatinTokenizer(vocab_file_path, encoder)
model_labert = AutoModel.from_pretrained(base_path)

In [11]:
# Models & tokenizers (base vs v1 fine-tuned)
# Adjust paths to your checkpoints
XLMR_ID_BASE   = "xlm-roberta-base"
XLMR_ID_V1     = "../data/models/xmlr_wsd/v1_cpu"     # your saved folder

LABERT_BASE_DIR= "/srv/models/latin-bert"
LABERT_V1_DIR  = "../data/models/labert_wsd/v1_cpu"   # your saved folder

In [13]:
# XLM-R base & v1
tok_xlmr_base = AutoTokenizer.from_pretrained(XLMR_ID_BASE)
mdl_xlmr_base = AutoModel.from_pretrained(XLMR_ID_BASE).to(DEVICE).eval()

tok_xlmr_v1   = AutoTokenizer.from_pretrained(XLMR_ID_V1)
mdl_xlmr_v1   = AutoModel.from_pretrained(XLMR_ID_V1).to(DEVICE).eval()

# LaBERT base & v1 (your custom tokenizer class already imported as LatinTokenizer)
tok_labert_base = tokenizer_labert  # if you kept the loaded instance; else re-instantiate here
mdl_labert_base = AutoModel.from_pretrained(LABERT_BASE_DIR).to(DEVICE).eval()

tok_labert_v1   = tokenizer_labert
mdl_labert_v1   = AutoModel.from_pretrained(LABERT_V1_DIR).to(DEVICE).eval()

In [28]:
import numpy as np
import torch

def _ensure_list(x):
    # spans may arrive as numpy arrays
    return [] if x is None else (x.tolist() if hasattr(x, "tolist") else list(x))

def _as_token_list(x):
    # Accept list/tuple/numpy array/Series; flatten to list[str]
    if x is None:
        return []
    if hasattr(x, "tolist"):           # numpy array / Series
        x = x.tolist()
    if isinstance(x, (tuple, set)):
        x = list(x)
    if isinstance(x, str):
        # If it's a stringified list like "['a','b']", try literal_eval; else split on whitespace
        import ast
        try:
            maybe = ast.literal_eval(x)
            if isinstance(maybe, (list, tuple)):
                x = list(maybe)
            else:
                x = x.split()
        except Exception:
            x = x.split()
    # ensure strings
    return [str(t) for t in x]


@torch.no_grad()
def all_layer_target_embedding_single(
    model, tokenizer, *,
    model_kind: str,      # "xlmr" | "labert"
    tokens,               # list[str] (surface or lemma tokens you already built)
    span,                 # list[int] subword indices for the target
    max_len: int = 256,
    device: str = "cpu",
    piece_pool: str = "mean",  # "mean" | "sum"
):
    """
    Returns an array of shape [num_encoder_layers, hidden_dim].
    Excludes the embeddings layer: layer 0 in the output = encoder layer 1 in HF.
    If span is empty -> returns None.
    """
    span = _ensure_list(span)
    if len(span) == 0:
        return None

    # Encode once
    if model_kind == "xlmr":
        tokens = _as_token_list(tokens)
        enc = tokenizer(tokens, is_split_into_words=True, return_tensors="pt",
                        truncation=True, padding="max_length", max_length=max_len)
    else:
        enc = tokenizer(" ".join(tokens), return_tensors="pt",
                        truncation=True, padding="max_length", max_length=max_len)

    enc = {k: v.to(device) for k, v in enc.items() if isinstance(v, torch.Tensor)}
    model = model.to(device).eval()

    outs = model(**enc, output_hidden_states=True, return_dict=True)
    hiddens = outs.hidden_states  # tuple length = num_layers+1 (incl. embeddings)

    idx = torch.tensor(span, device=device, dtype=torch.long)

    # We skip hiddens[0] (embeddings) and pool over encoder layers 1..N
    pooled = []
    for L in range(1, len(hiddens)):
        H = hiddens[L]            # [1, seq_len, D]
        vec = H[0, idx, :]        # [k, D]
        vec = vec.mean(dim=0) if piece_pool == "mean" else vec.sum(dim=0)
        pooled.append(vec.cpu().numpy())

    return np.stack(pooled, axis=0)   # [num_encoder_layers, D]

In [29]:
def all_layer_embeddings_from_df(
    df, model, tokenizer, *,
    model_kind: str,          # "xlmr" | "labert"
    view: str = "lemma",      # "lemma" | "surface"
    max_len: int = 256,
    device: str = "cpu",
    piece_pool: str = "mean",
):
    """
    Returns a list with one item per row:
      - None if span empty
      - or a np.ndarray of shape [num_encoder_layers, hidden_dim]
    Column names assumed from your build:
      tokens_lemma / tokens
      xmlr_wp_span_lemma / labert_wp_span_lemma  (or *_surface)
    """
    tokens_col = "tokens_lemma" if view == "lemma" else "tokens"
    span_col   = {"xlmr": f"xmlr_wp_span_{view}",
                  "labert": f"labert_wp_span_{view}"}[model_kind]

    out = []
    for r in df.itertuples(index=False):
        toks = _as_token_list(getattr(r, tokens_col))
        span = getattr(r, span_col)
        M = all_layer_target_embedding_single(
            model, tokenizer,
            model_kind=model_kind,
            tokens=toks,
            span=span,
            max_len=max_len,
            device=device,
            piece_pool=piece_pool,
        )
        out.append(M)
    return out

In [23]:
def average_layers(layer_stack: np.ndarray, layers, index_base: int = 0, reduce: str = "mean"):
    """
    layer_stack: np.ndarray [num_encoder_layers, D] from all_layer_* (encoder layers only)
    layers: int | Iterable[int] | (start, end) inclusive — indices refer to *encoder* layers.
            If index_base=1, pass 1-based and it will shift to 0-based internally.
    reduce: "mean" | "sum"
    """
    if layer_stack is None:
        return None

    # normalize layers input
    if isinstance(layers, int):
        Ls = [layers]
    elif isinstance(layers, tuple) and len(layers) == 2:
        a, b = layers
        Ls = list(range(a, b + 1))
    else:
        Ls = list(layers)

    if index_base == 1:
        Ls = [L - 1 for L in Ls]

    Ls = sorted(set(Ls))
    sub = layer_stack[Ls, :]    # [k, D]
    return sub.mean(0) if reduce == "mean" else sub.sum(0)

In [24]:
# 1) Extract all layers (lemma view) once
allvecs_labert_base_lemma = all_layer_embeddings_from_df(
    semeval_holdout, mdl_labert_base, tok_labert_base,
    model_kind="labert", view="lemma",
    max_len=256, device="cpu", piece_pool="mean"
)  # list of None or [num_layers, D]

allvecs_labert_v1_lemma = all_layer_embeddings_from_df(
    semeval_holdout, mdl_labert_v1, tok_labert_v1,
    model_kind="labert", view="lemma",
    max_len=256, device="cpu", piece_pool="mean"
)  # list of None or [num_layers, D]

In [30]:
allvecs_xlmr_base_lemma = all_layer_embeddings_from_df(
    semeval_holdout, mdl_xlmr_base, tok_xlmr_base,
    model_kind="xlmr", view="lemma",
    max_len=256, device="cpu", piece_pool="mean"
)

allvecs_xlmr_v1_lemma = all_layer_embeddings_from_df(
    semeval_holdout, mdl_xlmr_v1, tok_xlmr_v1,
    model_kind="xlmr", view="lemma",
    max_len=256, device="cpu", piece_pool="mean"
)

XLMRobertaSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


In [10]:
@torch.no_grad()
def get_hidden_all_layers(model, tokenizer, tokens, span, *, model_kind, max_len=256, device=DEVICE):
    """
    tokens: list[str] (surface or lemma view)
    span:   list[int] indices of target subword pieces in the final sequence
    Returns: [num_layers+1, hidden_size] pooled over span for every layer
    """
    if not span:  # guard
        return None

    if model_kind == "xlmr":
        enc = tokenizer(tokens, is_split_into_words=True, return_tensors="pt",
                        truncation=True, padding="max_length", max_length=max_len)
    else:  # labert – feed as a single string
        enc = tokenizer(" ".join(tokens), return_tensors="pt",
                        truncation=True, padding="max_length", max_length=max_len)

    enc = {k: v.to(device) for k, v in enc.items() if isinstance(v, torch.Tensor)}
    outs = model(**enc, output_hidden_states=True, return_dict=True)
    hiddens = outs.hidden_states  # tuple length = num_layers+1

    idx = torch.tensor(span, device=device, dtype=torch.long)
    pooled = []
    for H in hiddens:
        vec = H[0, idx, :].mean(dim=0)          # mean over target span
        pooled.append(vec.cpu().numpy())
    return np.stack(pooled, axis=0)              # [L+1, D]

In [ ]:
import numpy as np
import torch

def _ws(s): return s.strip().split()

def _tokens_and_target_idx(left: str, target: str, right: str):
    lt = _ws(left); rt = _ws(right)
    return lt + [target] + rt, len(lt)

def _span_xlmr(tokenizer, tokens, target_idx, max_len):
    enc = tokenizer(tokens, is_split_into_words=True, return_tensors="pt",
                    truncation=True, padding="max_length", max_length=max_len)
    word_ids = enc.word_ids(0)
    span = [] if word_ids is None else [i for i, wid in enumerate(word_ids) if wid == target_idx]
    return enc, span

def _span_labert(tokenizer, tokens, target_idx, max_len):
    # subword length per whitespace token (no specials)
    wp_lens = []
    for tok in tokens:
        try:
            ids = tokenizer.encode(tok, add_special_tokens=False)
        except TypeError:
            ids = tokenizer.encode(tok)
        wp_lens.append(len(ids))

    # encode full sentence for masks/length
    sent_str = " ".join(tokens)
    enc = tokenizer(sent_str, return_tensors="pt",
                    truncation=True, padding="max_length", max_length=max_len)

    # estimate left specials (CLS) robustly
    try:
        core_ids = tokenizer.encode(sent_str, add_special_tokens=False)
    except TypeError:
        core_ids = tokenizer.encode(sent_str)
    try:
        built = tokenizer.build_inputs_with_special_tokens(core_ids)
        def _first(hay, ned):
            for i in range(0, len(hay)-len(ned)+1):
                if hay[i:i+len(ned)] == ned: return i
            return None
        start = _first(built, core_ids)
        left_specials = start if start is not None else 1
    except Exception:
        left_specials = 1

    start_in_core = sum(wp_lens[:target_idx])
    k = wp_lens[target_idx] if target_idx < len(wp_lens) else 0
    if k <= 0:
        return enc, []
    start = left_specials + start_in_core
    end = start + k - 1
    seq_len = int(enc["attention_mask"][0].sum().item())
    span = [i for i in range(start, end+1) if 0 <= i < seq_len]
    return enc, span

In [14]:
@torch.no_grad()
def target_embedding(
    model, tokenizer,
    *, model_kind: str,       # "xlmr" or "labert"
    left: str, target_surface: str, target_lemma: str, right: str,
    view: str = "lemma",      # "lemma" or "surface"
    layer_idx: int = 9,
    max_len: int = 256,
    device: str = "cpu",
    pool: str = "mean",       # "mean" or "sum"
):
    # choose which token to inject at target position
    target = target_lemma if view == "lemma" else target_surface
    tokens, tidx = _tokens_and_target_idx(left, target, right)

    # tokenize & locate target span
    if model_kind == "xlmr":
        enc, span = _span_xlmr(tokenizer, tokens, tidx, max_len)
    else:
        enc, span = _span_labert(tokenizer, tokens, tidx, max_len)

    if not span:
        return None  # can't pool if span is empty

    # to device & forward
    enc = {k: v.to(device) for k, v in enc.items() if isinstance(v, torch.Tensor)}
    model = model.to(device).eval()
    outs = model(**enc, output_hidden_states=True, return_dict=True)
    H = outs.hidden_states[layer_idx]        # [1, L, D]

    idx = torch.tensor(span, device=device, dtype=torch.long)
    vec = H[0, idx, :].mean(dim=0) if pool == "mean" else H[0, idx, :].sum(dim=0)
    return vec.detach().cpu().numpy()        # np.ndarray [D]

In [17]:
import numpy as np
import torch

def _normalize_layers_arg(layers, *, index_base=0):
    """
    layers: int | Iterable[int] | Tuple[int, int]  (inclusive range)
    index_base: 0 (HF default) or 1 (if you prefer 1-based inputs)
    Returns sorted unique 0-based layer indices (list[int]).
    Note: hidden_states[0] is embeddings; encoder layers start at 1.
    """
    if isinstance(layers, int):
        Ls = [layers]
    elif isinstance(layers, tuple) and len(layers) == 2:
        a, b = layers
        Ls = list(range(a, b + 1))
    else:
        Ls = list(layers)

    # shift if user passes 1-based
    if index_base == 1:
        Ls = [L - 1 for L in Ls]
    # ensure unique, sorted
    Ls = sorted(set(Ls))
    return Ls

@torch.no_grad()
def target_embedding(
    model, tokenizer,
    *, model_kind: str,           # "xlmr" or "labert"
    left: str, target_surface: str, target_lemma: str, right: str,
    view: str = "lemma",          # "lemma" or "surface"
    layers=9,                     # int, list/tuple of ints, or (start,end) inclusive
    index_base: int = 0,          # 0-based (Labyrinthus uses this), set 1 if you pass 1-based
    max_len: int = 256,
    device: str = "cpu",
    piece_pool: str = "mean",     # "mean" | "sum" over subword pieces
    layer_pool: str = "mean",     # "mean" | "sum" across *layers*
):
    """
    Returns a single vector [D] for the target span.
    If 'layers' is multiple, pools across those layers with `layer_pool`.
    """
    def _ws(s): return s.strip().split()
    def _tokens_and_target_idx(L, T, R):
        lt = _ws(L); rt = _ws(R)
        return lt + [T] + rt, len(lt)

    # build tokens
    target = target_lemma if view == "lemma" else target_surface
    tokens, tidx = _tokens_and_target_idx(left, target, right)

    # locate span per tokenizer
    if model_kind == "xlmr":
        enc = tokenizer(tokens, is_split_into_words=True, return_tensors="pt",
                        truncation=True, padding="max_length", max_length=max_len)
        word_ids = enc.word_ids(0)
        span = [] if word_ids is None else [i for i, wid in enumerate(word_ids) if wid == tidx]
    else:
        # LaBERT: subsequence-based span (same logic you already used)
        wp_lens = []
        for tok in tokens:
            try:
                ids = tokenizer.encode(tok, add_special_tokens=False)
            except TypeError:
                ids = tokenizer.encode(tok)
            wp_lens.append(len(ids))
        sent_str = " ".join(tokens)
        enc = tokenizer(sent_str, return_tensors="pt",
                        truncation=True, padding="max_length", max_length=max_len)
        # estimate left specials
        try:
            core_ids = tokenizer.encode(sent_str, add_special_tokens=False)
        except TypeError:
            core_ids = tokenizer.encode(sent_str)
        left_specials = 1
        try:
            built = tokenizer.build_inputs_with_special_tokens(core_ids)
            def _first(hay, ned):
                for i in range(0, len(hay) - len(ned) + 1):
                    if hay[i:i+len(ned)] == ned: return i
                return None
            start = _first(built, core_ids)
            if start is not None: left_specials = start
        except Exception:
            pass

        start_in_core = sum(wp_lens[:tidx])
        k = wp_lens[tidx] if tidx < len(wp_lens) else 0
        if k <= 0:
            return None
        start = left_specials + start_in_core
        end = start + k - 1
        seq_len = int(enc["attention_mask"][0].sum().item())
        span = [i for i in range(start, end + 1) if 0 <= i < seq_len]

    if not span:
        return None

    # forward & gather layers
    enc = {k: v.to(device) for k, v in enc.items() if isinstance(v, torch.Tensor)}
    model = model.to(device).eval()
    outs = model(**enc, output_hidden_states=True, return_dict=True)
    hiddens = outs.hidden_states  # tuple len = num_layers+1

    idx = torch.tensor(span, device=device, dtype=torch.long)
    layer_ids = _normalize_layers_arg(layers, index_base=index_base)

    # pool over pieces per selected layer
    per_layer = []
    for L in layer_ids:
        H = hiddens[L]  # [1, Lseq, D]
        vec = H[0, idx, :]
        vec = vec.mean(dim=0) if piece_pool == "mean" else vec.sum(dim=0)
        per_layer.append(vec)
    M = torch.stack(per_layer, dim=0)  # [n_layers, D]

    # pool across layers
    out = M.mean(dim=0) if layer_pool == "mean" else M.sum(dim=0)
    return out.detach().cpu().numpy()


def target_embeddings_from_df(
    df, model, tokenizer, *,
    model_kind: str,            # "xlmr" | "labert"
    view: str = "lemma",
    layers=9,                   # int | list | (start,end)
    index_base: int = 0,
    max_len: int = 256,
    device: str = "cpu",
    piece_pool: str = "mean",
    layer_pool: str = "mean",
):
    tokens_col = "tokens_lemma" if view == "lemma" else "tokens"
    span_col   = {"xlmr": f"xmlr_wp_span_{view}",
                  "labert": f"labert_wp_span_{view}"}[model_kind]

    model = model.to(device).eval()
    layer_ids = _normalize_layers_arg(layers, index_base=index_base)
    out = []

    with torch.no_grad():
        for r in df.itertuples(index=False):
            toks = getattr(r, tokens_col)
            span = getattr(r, span_col)

            # --- FIX: handle numpy arrays or lists ---
            if span is None or len(span) == 0:
                out.append(None)
                continue
            span = list(span)   # force into Python list

            if model_kind == "xlmr":
                enc = tokenizer(toks, is_split_into_words=True, return_tensors="pt",
                                truncation=True, padding="max_length", max_length=max_len)
            else:
                enc = tokenizer(" ".join(toks), return_tensors="pt",
                                truncation=True, padding="max_length", max_length=max_len)
            enc = {k: v.to(device) for k, v in enc.items() if isinstance(v, torch.Tensor)}

            outs = model(**enc, output_hidden_states=True, return_dict=True)
            hiddens = outs.hidden_states

            idx = torch.tensor(span, device=device, dtype=torch.long)

            per_layer = []
            for L in layer_ids:
                H = hiddens[L]  # [1, Lseq, D]
                vec = H[0, idx, :]
                vec = vec.mean(dim=0) if piece_pool == "mean" else vec.sum(dim=0)
                per_layer.append(vec)

            M = torch.stack(per_layer, dim=0)         # [n_layers, D]
            v = M.mean(dim=0) if layer_pool == "mean" else M.sum(dim=0)
            out.append(v.detach().cpu().numpy())

    return out

In [18]:
vecs = target_embeddings_from_df(semeval_holdout, mdl_labert_v1, tok_labert_v1,
                                 model_kind="labert", view="lemma",
                                 layers=9, index_base=0)

In [20]:
len(vecs)

480